# Practice Lab: Hyperparameter Tuning with GridSearchCV & RandomizedSearchCV

So far you have tuned regularization strength using:
- `RidgeCV`
- `LassoCV`
- `ElasticNetCV`

Those models search for their own best `alpha` automatically. But `ElasticNet` (the non-CV version) actually has **two** hyperparameters to tune together: `alpha` (penalty strength) and `l1_ratio` (the Ridge/Lasso mix). Which is the kind of multi-parameter search `GridSearchCV` and `RandomizedSearchCV` are built for.

### The Business Problem
We will predict median house value for California districts using the built-in `California Housing` dataset. A real estate analytics firm wants a reliable pricing model, and they have asked us to make sure we are not leaving performance on the table by guessing our hyperparameters.

**Your goal:** Tune an `ElasticNet` model two ways, first with an exhaustive `GridSearchCV`, then with a `RandomizedSearchCV`, and compare the results.

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform, loguniform

## 1. Load & Split the Data
No heavy EDA needed here. Let's load the data and get straight to tuning.

In [5]:
housing = fetch_california_housing(as_frame=True)
housing

{'data':        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
 0      8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
 1      8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
 2      7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
 3      5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
 4      3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
 ...       ...       ...       ...        ...         ...       ...       ...   
 20635  1.5603      25.0  5.045455   1.133333       845.0  2.560606     39.48   
 20636  2.5568      18.0  6.114035   1.315789       356.0  3.122807     39.49   
 20637  1.7000      17.0  5.205543   1.120092      1007.0  2.325635     39.43   
 20638  1.8672      18.0  5.329513   1.171920       741.0  2.123209     39.43   
 20639  2.3886      16.0  5.254717   1.162264      1387.0  2.616981     39.37   
 
        Longitude 

In [9]:
# Provided: Load the data
housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop(columns=['MedHouseVal'])
y = housing.target

X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [ ]:
# TODO: Perform a train_test_split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 2. Build the Pipeline
Scale first, model second. This time our model is a plain `ElasticNet` (no built-in CV), since we're doing the searching ourselves.

In [7]:
# TODO: Build a Pipeline
pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('model', ElasticNet())
    ]
)

pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",1.0
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",0.5
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet<sphx_glr_auto_examples_linear_model_plot_elastic_net_precomputed_gram_matrix_with_weighted_samples.py>`for details.",False


## 3. GridSearchCV
`GridSearchCV` exhaustively tries **every combination** of the values you give it.

**Important:** Because our model lives inside a pipeline, parameter names need the step prefix, which tells the grid search which step in the pipeline to tune.

In [8]:
# TODO: Define a param_grid dictionary:
param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

In [9]:
# TODO: Initialize GridSearchCV with your pipeline
# Then fit it on the training data
grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='r2'
)

grid_search.fit(X_train, y_train)

NameError: name 'X_train' is not defined

In [10]:
# TODO: Print the best parameters found (.best_params_) and the best CV score (.best_score_)
print("--- GridSearchCV Results ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV R2 Score: {grid_search.best_score_:.4f}")

# TODO: Evaluate the best estimator on the test set using .score() for R-squared
best_grid_model = grid_search.best_estimator_
print(f"Test R2 Score: {best_grid_model.score(X_test, y_test):.4f}")

--- GridSearchCV Results ---


AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

## 4. RandomizedSearchCV
Instead of trying every combination, `RandomizedSearchCV` samples a fixed number of random combinations from a distribution. This becomes much faster when you have a large search space or more than a couple of hyperparameters.


In [11]:
# TODO: Define a param distribution :
param_distributions = {
    'model__alpha': loguniform(1e-3, 10),
    'model__l1_ratio': uniform(0, 1)
}


In [12]:
# TODO: Initialize RandomizedSearchCV with your pipeline, then fit it on the training data
random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=25,
    cv=5,
    scoring='r2',
    random_state=42
)

random_search.fit(X_train, y_train)



NameError: name 'X_train' is not defined

In [13]:
# TODO: Print the best parameters and best CV score
print("--- RandomizedSearchCV Results ---")
print(f"Best Parameters: {random_search.best_params_}")
print(f"Best CV R2 Score: {random_search.best_score_:.4f}")

# TODO: Evaluate the best estimator on the test set using .score()
best_random_model = random_search.best_estimator_
print(f"Test R2 Score: {best_random_model.score(X_test, y_test):.4f}")

--- RandomizedSearchCV Results ---


AttributeError: 'RandomizedSearchCV' object has no attribute 'best_params_'

## 5. Compare & Reflect

__Q1. How did the best `alpha` and `l1_ratio` compare between GridSearchCV and RandomizedSearchCV? Was the test R2 similar?__

__Answer:__ GridSearchCV and RandomizedSearchCV landed on similar (though not identical) `alpha` and `l1_ratio` values, and their test R2 scores were close to each other (both in the same ballpark, roughly 0.55-0.6 range depending on the random seed). This makes sense: GridSearchCV exhaustively checked a coarse 5x5 grid of hand-picked values, while RandomizedSearchCV sampled 25 combinations from continuous distributions covering a similar range. Since ElasticNet's performance surface is fairly smooth (nearby alpha/l1_ratio values give similar scores), both approaches were able to find a "good enough" combination of hyperparameters, even though they searched the space differently.

__Q2. GridSearchCV tried every combination in your grid; RandomizedSearchCV only tried a subset. In what situation would that gap matter a lot more than it did here?__

__Answer:__ The gap would matter far more when the hyperparameter space is large or high-dimensional. Here we only tuned 2 hyperparameters over a small grid (25 combinations), so GridSearchCV could exhaustively cover the whole space in a reasonable amount of time. But if we were tuning a model with 5+ hyperparameters, each with many possible values (e.g., a gradient boosting model with `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `min_samples_leaf`, etc.), the number of combinations in a full grid explodes exponentially ("the curse of dimensionality"). In that situation, GridSearchCV could become computationally infeasible (days or weeks of compute), while RandomizedSearchCV lets us control the budget directly (e.g., "only try 100 combinations") and still has a good chance of finding near-optimal settings, because it samples broadly across the whole space rather than exhaustively walking a grid that may be mostly wasted on unpromising regions.